# Example: Run experiments from notebook

This notebook is an example of how to run the pipeline from Jupyter and inspect outputs.

It demonstrates:
1. robust repo-root detection
2. discovering configured providers/models
3. run with provider-only override (`openai`, all configured OpenAI models)
4. run with exact provider/model pair (`openai/gpt-4o`)
5. batch run across all available provider/model pairs


In [1]:
from pathlib import Path
import json
import os

from text2sql_eval import build_rag_index, run_experiment
from text2sql_eval.config import load_config

def detect_repo_root(start: Path) -> Path:
    env_root = os.getenv("TEXT2SQL_EVAL_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "pyproject.toml").exists() and (candidate / "config/config.yaml").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "config/config.yaml").exists():
            return candidate

    sibling = start / "text2sql-eval"
    if (sibling / "pyproject.toml").exists() and (sibling / "config/config.yaml").exists():
        return sibling

    raise RuntimeError(
        f"Could not detect repo root from {start}. "
        "Set TEXT2SQL_EVAL_ROOT=/absolute/path/to/text2sql-eval"
    )

REPO_ROOT = detect_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)

CONFIG_PATH = REPO_ROOT / "config/config.yaml"
cfg = load_config(str(CONFIG_PATH))

print("repo:", REPO_ROOT)
print("config:", CONFIG_PATH)
print("questions:", REPO_ROOT / cfg.inputs.questions_file)
print("database:", REPO_ROOT / cfg.inputs.database_file)


repo: /home/mileto/dev/text2sql-eval
config: /home/mileto/dev/text2sql-eval/config/config.yaml
questions: /home/mileto/dev/text2sql-eval/data/dev.json
database: /home/mileto/dev/text2sql-eval/data/database.sqlite


## Optional: build the RAG index

Run this before using `track="c"`. It indexes the curated documents in `docs/rag/` into Chroma using the RAG settings from `config/config.yaml`.

Requirements:
- `OPENAI_API_KEY` set in the notebook environment
- curated documents already present under `docs/rag/`


In [2]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before building the RAG index")

rag_result = build_rag_index(config_path=str(CONFIG_PATH))
print("RAG index:", rag_result.index_path)
print("manifest:", rag_result.manifest_path)
print("source files:", rag_result.source_count)
print("chunks:", rag_result.chunk_count)


RAG index: data/rag_index
manifest: data/rag_index/manifest.json
source files: 5
chunks: 21


In [3]:
provider_models: dict[str, list[str]] = {}
for model_cfg in cfg.models:
    provider_models.setdefault(model_cfg.provider, []).append(model_cfg.model)

print("Configured provider/model pairs:")
for provider, models in provider_models.items():
    print(f"- {provider}: {models}")


Configured provider/model pairs:
- fake: ['local-test']
- openai: ['gpt-4o']
- anthropic: ['claude-3-5-sonnet-20241022']


## Example A: run only OpenAI provider (all configured OpenAI models)

In [4]:
run_id_openai = run_experiment(
    config_path=str(CONFIG_PATH),
    track="c",
    limit=3,
    provider="openai",
)

artifact_openai = REPO_ROOT / cfg.run_defaults.output_dir / run_id_openai / "run.json"
print("run_id_openai:", run_id_openai)
print("artifact:", artifact_openai)


run_id_openai: 20260404-213241
artifact: /home/mileto/dev/text2sql-eval/results/20260404-213241/run.json


## Example B: run only one model (`openai/gpt-4o`)

In [ ]:
run_id_gpt4o = run_experiment(
    config_path=str(CONFIG_PATH),
    track="a",
    limit=3,
    provider="openai",
    model="gpt-4o",
)

artifact_gpt4o = REPO_ROOT / cfg.run_defaults.output_dir / run_id_gpt4o / "run.json"
print("run_id_gpt4o:", run_id_gpt4o)
print("artifact:", artifact_gpt4o)


## Example C (last cell): batch run all available provider/model pairs

In [ ]:
def provider_is_available(provider: str) -> bool:
    if provider == "fake":
        return True
    if provider == "openai":
        return bool(os.getenv("OPENAI_API_KEY"))
    if provider == "anthropic":
        return bool(os.getenv("ANTHROPIC_API_KEY"))
    return False

TRACK = "a"
LIMIT = 3

run_ids: dict[str, str] = {}
skipped: list[str] = []

for provider, models in provider_models.items():
    for model in models:
        pair = f"{provider}/{model}"
        if not provider_is_available(provider):
            skipped.append(pair)
            continue

        run_id = run_experiment(
            config_path=str(CONFIG_PATH),
            track=TRACK,
            limit=LIMIT,
            provider=provider,
            model=model,
        )
        run_ids[pair] = run_id

print("Completed runs:")
for pair, run_id in run_ids.items():
    artifact = REPO_ROOT / cfg.run_defaults.output_dir / run_id / "run.json"
    print(f"- {pair}: {artifact}")

if skipped:
    print("\nSkipped (missing credentials):")
    for pair in skipped:
        print(f"- {pair}")
